# Multi-Stage Queries in SLayer

Most semantic layers parameterize a single GROUP BY. But what if you need to aggregate *an aggregation* — like averaging monthly revenue across stores, or bucketing users by a computed metric?

SLayer handles this naturally: every query implies a model (via auto-introspection of its result columns), so you can chain queries together. This notebook demonstrates two multi-stage patterns with working code.

See also: [Query Lists](../../concepts/queries.md#query-lists) | [ModelExtension](../../concepts/queries.md#modelextension) | [Creating Models from Queries](../../concepts/models.md#creating-models-from-queries)

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Queries as Models

In most semantic layers, queries and models are entirely separate concepts. In SLayer, every query resolves to SQL, and SLayer's [auto-introspection](../../concepts/ingestion.md) can generate a model from any SQL SELECT query. Put these together: **every query automatically implies a model**.

When a query becomes a virtual model, its result columns become dimensions, and numeric columns also get auto-generated aggregation measures (`_sum`, `_avg`, `_min`, `_max`, `_distinct`). This means you can use one query's output as the input to another — no special syntax needed.

## Example 1: Average Monthly Revenue per Store

**Goal:** What is the average monthly revenue for each store?

This is a two-stage query:
1. **Inner query:** Total revenue grouped by store and month
2. **Outer query:** Average the monthly revenue, grouped by store

The inner query's `order_total_sum` measure becomes a dimension on the virtual model, which automatically gets an `order_total_sum_avg` measure we can use in the outer query.

In [2]:
# Stage 1: Monthly revenue by store
inner_result = engine.execute(query={
    "source_model": "orders",
    "fields": ["order_total_sum"],
    "dimensions": ["stores.name"],
    "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
    "order": [{"column": "ordered_at", "direction": "asc"}],
    "limit": 10,
})

print("Inner query result (first 10 rows):")
print(f"{'Month':<12} {'Store':<20} {'Revenue':>12}")
print("-" * 46)
for row in inner_result.data:
    month = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} {store:<20} ${rev:>11,.2f}")

Inner query result (first 10 rows):
Month        Store                     Revenue
----------------------------------------------
2018-09      Philadelphia         $   4,738.80
2018-10      Philadelphia         $   7,882.42
2018-11      Philadelphia         $  11,394.11
2018-12      Philadelphia         $  14,904.61
2019-01      Philadelphia         $  17,790.62
2019-02      Philadelphia         $  17,464.28
2019-03      Brooklyn             $  10,592.40
2019-03      Philadelphia         $  20,683.17
2019-04      Brooklyn             $  15,126.80
2019-04      Philadelphia         $  21,322.16


Now we chain the two queries together. We give the inner query a `name` and pass both queries as a list to `engine.execute()`. The outer query references the inner query's name as its `source_model`:

In [3]:
# Two-stage query: average monthly revenue per store
result = engine.execute(query=[
    {
        "name": "monthly_store_revenue",
        "source_model": "orders",
        "fields": ["order_total_sum"],
        "dimensions": ["stores.name"],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
    },
    {
        "source_model": "monthly_store_revenue",
        "fields": ["order_total_sum_avg"],
        "dimensions": ["stores__name"],
        "order": [{"column": "order_total_sum_avg", "direction": "desc"}],
    },
])

print("Average monthly revenue by store:")
print(f"{'Store':<20} {'Avg Monthly Revenue':>20}")
print("-" * 42)
for row in result.data:
    store = row["monthly_store_revenue.stores__name"]
    avg_rev = row["monthly_store_revenue.order_total_sum_avg"]
    print(f"{store:<20} ${avg_rev:>19,.2f}")

Average monthly revenue by store:
Store                 Avg Monthly Revenue
------------------------------------------
Brooklyn             $          33,826.56
Chicago              $          26,601.08
San Francisco        $          23,323.39
Philadelphia         $          20,941.13
New Orleans          $           9,251.65


**What happened under the hood:**

1. SLayer executed the inner query (`monthly_store_revenue`), producing rows of (store, month, revenue)
2. It auto-introspected those results to create a virtual model with:
   - Dimensions: `stores__name`, `ordered_at`, `order_total_sum`
   - Measures: `order_total_sum_sum`, `order_total_sum_avg`, `order_total_sum_min`, `order_total_sum_max`, `count`, etc.
3. The outer query used this virtual model, grouping by `stores__name` and requesting `order_total_sum_avg`

The virtual model is a self-contained table — it no longer has the joins the source model had. That's why the dimension is called `stores__name` (using `__` to encode the original join path) rather than `stores.name` (which would imply a join to a `stores` model that doesn't exist on the virtual model).

No special multi-stage syntax was needed — just a named query and a reference to it.

In [4]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  monthly_store_revenue.stores__name AS "monthly_store_revenue.stores__name",
  AVG(monthly_store_revenue.order_total_sum) AS "monthly_store_revenue.order_total_sum_avg"
FROM (
  SELECT
    "orders.stores.name" AS stores__name,
    "orders.ordered_at" AS ordered_at,
    "orders.order_total_sum" AS order_total_sum
  FROM (
    SELECT
      stores.name AS "orders.stores.name",
      DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
      SUM(orders.order_total) AS "orders.order_total_sum"
    FROM orders AS orders
    LEFT JOIN stores AS stores
      ON orders.store_id = stores.id
    GROUP BY
      stores.name,
      DATE_TRUNC('MONTH', orders.ordered_at)
  ) AS _inner
) AS monthly_store_revenue
GROUP BY
  monthly_store_revenue.stores__name
ORDER BY
  "monthly_store_revenue.order_total_sum_avg" DESC


## Example 2: Orders by Customer Activity Bucket

**Goal:** How many orders and how much revenue comes from low, medium, and high-activity customers?

This is a two-stage pattern with a join:
1. **Inner query:** Compute total order count per customer
2. **Outer query:** Join the inner result to `orders`, add a CASE-based bucket dimension via [ModelExtension](../../concepts/queries.md#modelextension), and group by the bucket

The bucketing uses an [inline dimension](../../concepts/queries.md#modelextension) — a CASE expression that categorizes each customer's total order count into Low / Medium / High.

In [5]:
# Stage 1: Total orders per customer
inner_result = engine.execute(query={
    "source_model": "orders",
    "fields": ["count"],
    "dimensions": ["customer_id"],
    "order": [{"column": {"name": "count"}, "direction": "desc"}],
    "limit": 10,
})

print("Top 10 customers by order count:")
print(f"{'Customer ID':<40} {'Orders':>10}")
print("-" * 52)
for row in inner_result.data:
    cust = row["orders.customer_id"]
    count = row["orders.count"]
    print(f"{cust:<40} {count:>10,}")

Top 10 customers by order count:
Customer ID                                  Orders
----------------------------------------------------
cb05c633-6565-481c-bd3b-c5ea4698b3a4            560
5bf39e4b-ecc5-495e-bd8c-175fe8ae2f9e            537
233c7e68-aadd-4646-b437-51657c5e9740            536
a0a6321c-df4f-4f4c-8e7d-293d033b98d0            514
3bfb5bc9-a3da-401d-99d0-be347d8a8a70            513
5e9fb5f3-27ac-4b55-9b24-487144ce4c0f            509
c2dcc5e3-610f-45a4-9346-8f7949873ee4            507
547163ac-354c-40e2-985d-26084b0a4609            491
b68b5bc3-d727-4665-87f1-39281aa1e477            490
000693d3-847c-4b72-85ba-6650ad9265a7            489


Now we chain the queries. The outer query uses `ModelExtension` to:
1. **Join** the inner query (customer order counts) to the `orders` model via `customer_id`
2. **Define an inline dimension** using a CASE expression that buckets customers by their total order count

Since the bucket dimension references a column from the joined sub-query, its SQL uses the joined table alias: `customer_activity.count` (the sub-query name followed by the column).

In [6]:
# Two-stage query: orders grouped by customer activity bucket
result = engine.execute(query=[
    {
        "name": "customer_activity",
        "source_model": "orders",
        "fields": ["count"],
        "dimensions": ["customer_id"],
    },
    {
        "source_model": {
            "source_name": "orders",
            "joins": [
                {
                    "target_model": "customer_activity",
                    "join_pairs": [["customer_id", "customer_id"]],
                }
            ],
            "dimensions": [
                {
                    "name": "activity_bucket",
                    "sql": "CASE WHEN customer_activity.count >= 500 THEN 'High' WHEN customer_activity.count >= 200 THEN 'Medium' ELSE 'Low' END",
                    "type": "string",
                }
            ],
        },
        "fields": ["count", "order_total_sum"],
        "dimensions": ["activity_bucket"],
        "order": [{"column": {"name": "order_total_sum"}, "direction": "desc"}],
    },
])

print("Orders by customer activity bucket:")
print(f"{'Bucket':<12} {'Orders':>10} {'Revenue':>14}")
print("-" * 38)
for row in result.data:
    bucket = row["orders.activity_bucket"]
    count = row["orders.count"]
    rev = row["orders.order_total_sum"]
    print(f"{bucket:<12} {count:>10,} ${rev:>13,.2f}")

Orders by customer activity bucket:
Bucket           Orders        Revenue
--------------------------------------
Low             105,225 $ 1,435,049.70
Medium           97,484 $ 1,170,118.78
High              3,676 $    44,411.39


In [7]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
SELECT
  CASE
    WHEN customer_activity.count >= 500
    THEN 'High'
    WHEN customer_activity.count >= 200
    THEN 'Medium'
    ELSE 'Low'
  END AS "orders.activity_bucket",
  COUNT(*) AS "orders.count",
  SUM(orders.order_total) AS "orders.order_total_sum"
FROM orders AS orders
LEFT JOIN (SELECT "orders.customer_id" AS customer_id, "orders.count" AS count FROM (SELECT
  orders.customer_id AS "orders.customer_id",
  COUNT(*) AS "orders.count"
FROM orders AS orders
GROUP BY
  orders.customer_id) AS _inner) AS customer_activity ON orders.customer_id = customer_activity.customer_id
GROUP BY
  CASE
    WHEN customer_activity.count >= 500
    THEN 'High'
    WHEN customer_activity.count >= 200
    THEN 'Medium'
    ELSE 'Low'
  END
ORDER BY
  "orders.order_total_sum" DESC


## Summary

Multi-stage queries in SLayer require no special syntax — they emerge naturally from two features:

| Feature | Role in Multi-Stage Queries |
|---------|----------------------------|
| **Query-as-model** | Every query's result auto-generates a virtual model with dimensions and measures |
| **Query lists** | Pass `[inner, outer]` to `engine.execute()` — inner queries are named, outer references them |
| **Auto-generated measures** | Numeric columns from inner queries get `_sum`, `_avg`, `_min`, `_max` measures automatically |
| **ModelExtension joins** | Join a sub-query's virtual model to any stored model at query time |
| **Inline dimensions** | Add CASE expressions or other computed dimensions via `ModelExtension` |

See the [Query Lists docs](../../concepts/queries.md#query-lists) and [ModelExtension docs](../../concepts/queries.md#modelextension) for the full reference.